# P07 SkyOps — Week 5: Bronze → Silver Candidate

### ZENAIZ × BVRIT Hyderabad Data Engineering Internship

**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 5 — Bronze to Silver Candidate  
**Technology:** Databricks + Spark SQL + Delta

## Week 5 objective

Transform the four completed SkyOps Bronze Delta tables into **Silver Candidate** Delta tables by applying only project-supported transformations:

- standardise controlled identifiers/categories;
- convert raw Bronze strings to documented Silver types using `TRY_CAST`;
- create the Silver `flight_id` from the stable `source_record_key`;
- preserve Bronze lineage and source provenance;
- prove that Bronze physical rows are preserved;
- retain parse failures for the later DQ stage.

```text
Bronze Delta tables
        ↓
Standardise + safe type conversion
        ↓
Silver Candidate Delta tables
        ↓
Count + lineage + schema validation
        ↓
Controlled rerun proof
```

### Important project boundary

The supplied SkyOps project pack provides the source manifest, data dictionary, provenance, and Week-6 DQ rules, but it does **not** define additional Week-5 calculated KPI formulas or a Week-5 reference-enrichment rule. Therefore this notebook does **not invent** such logic.

Week 5 also does **not** quarantine, deduplicate, repair invalid business facts, build Trusted Silver, build Gold, or process the Week-10 streaming drops.


## Why this corrected version is needed

The submitted Week-5 notebook had several problems that could make it fail or weaken the project contract:

1. It treated the project as if `_source_period` existed in `bronze_airports` and `bronze_routes`; the corrected Week 4 contract does not add that field to those reference tables.
2. It dropped most Bronze lineage fields from `silver_airports_candidate`.
3. It did not retain `source_record_key` separately from the derived `flight_id`.
4. It used `CAST(flight_date AS DATE)` instead of the Week-5 safe-cast pattern.
5. It did not provide a complete, consistent validation framework for all four entities.
6. It added transformations without a documented SkyOps Week-5 specification.

This notebook keeps the Week-5 method from the supplied learning material, but maps it only to fields supported by the SkyOps project files.


## Source and target contract

| Bronze input | Silver Candidate output | Supported Week-5 work |
|---|---|---|
| `bronze_airports` | `silver_airports_candidate` | trim/uppercase code fields, safe-cast `active_flag`, preserve lineage |
| `bronze_carriers` | `silver_carriers_candidate` | trim/uppercase code, safe-cast `active_flag`, preserve lineage |
| `bronze_routes` | `silver_routes_candidate` | standardise codes, safe-cast `distance_miles`, preserve lineage |
| `bronze_flights` | `silver_flights_candidate` | create `flight_id`, standardise codes, safe-cast documented numeric/date/flag fields, preserve source provenance and lineage |

### Project-supported target types

From the supplied data dictionary:

- dates → `DATE`
- flight numbers / flags → `INT`
- source row number → `INT`
- delay, duration and distance measures → `DOUBLE`
- route distance → `INT`
- identifiers/codes/source provenance → `STRING`

`TRY_CAST` is used so an unparseable Bronze value becomes `NULL` while the physical row remains present. Week 6 owns the DQ decision.


## How to run this notebook

Run **top to bottom, one executable cell at a time** in Databricks.

For each step:

1. Read the purpose.
2. Run the SQL cell.
3. Inspect the result.
4. Resolve any mismatch before continuing.

Do not type execution results into markdown. Counts, schemas and validation results must come from your actual Databricks run.


## 1. Confirm the Week-4 handoff


In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;


In [ ]:
%sql
SHOW TABLES;


In [ ]:
%sql
SELECT
  'airports' AS entity,
  COUNT(*) AS bronze_rows
FROM bronze_airports
UNION ALL
SELECT 'carriers', COUNT(*) FROM bronze_carriers
UNION ALL
SELECT 'routes', COUNT(*) FROM bronze_routes
UNION ALL
SELECT 'flights', COUNT(*) FROM bronze_flights
ORDER BY entity;


### Handoff checkpoint

The four production Bronze tables must exist:

- `bronze_airports`
- `bronze_carriers`
- `bronze_routes`
- `bronze_flights`

The actual counts are obtained from Databricks. The project manifest gives reference counts of 20, 10, 360 and 123,337 respectively, but this notebook does not hard-code those execution results.


In [ ]:
%sql
DESCRIBE bronze_airports;


In [ ]:
%sql
DESCRIBE bronze_carriers;


In [ ]:
%sql
DESCRIBE bronze_routes;


In [ ]:
%sql
DESCRIBE bronze_flights;


## 2. Understand the three Week-5 transformation patterns

### Pattern A — standardise controlled identifiers

Use `TRIM` and `UPPER` only for identifier/code fields where one canonical representation is appropriate. Do not arbitrarily uppercase free-text names or descriptions.

### Pattern B — safe type conversion

Use `TRY_CAST` rather than a hard cast for Bronze-to-Candidate conversion. This preserves the physical row when a value cannot be parsed.

### Pattern C — preserve lineage

Every Candidate row must retain the Bronze source file, source path, ingestion run, Bronze schema version, Bronze record hash and rescued/parser context. For flights, the source's own provenance fields must also remain available.


## 3. Build `silver_airports_candidate`


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_airports_candidate
USING DELTA
AS
SELECT
    UPPER(TRIM(airport_code)) AS airport_code,
    TRIM(airport_name) AS airport_name,
    TRIM(city) AS city,
    TRIM(state_region) AS state_region,
    TRY_CAST(active_flag AS INT) AS active_flag,

    -- Bronze lineage
    _source_row_number,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash,
    _rescued_payload,

    -- Silver Candidate metadata
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'skyops_silver_candidate_v1.0' AS _candidate_schema_version

FROM bronze_airports;


In [ ]:
%sql
SELECT
    airport_code,
    airport_name,
    city,
    state_region,
    active_flag,
    _source_file_name,
    _ingestion_run_id,
    _bronze_record_hash
FROM silver_airports_candidate
LIMIT 10;


In [ ]:
%sql
DESCRIBE silver_airports_candidate;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM bronze_airports) AS bronze_rows,
    (SELECT COUNT(*) FROM silver_airports_candidate) AS candidate_rows,
    (SELECT COUNT(*) FROM silver_airports_candidate)
      - (SELECT COUNT(*) FROM bronze_airports) AS difference,
    CASE
      WHEN (SELECT COUNT(*) FROM bronze_airports)
         = (SELECT COUNT(*) FROM silver_airports_candidate)
      THEN 'PASS' ELSE 'CHECK'
    END AS status;


In [ ]:
%sql
SELECT COUNT(*) AS active_flag_parse_failures
FROM bronze_airports
WHERE active_flag IS NOT NULL
  AND TRY_CAST(active_flag AS INT) IS NULL;


### Airports checkpoint

Expected:

- Candidate count equals Bronze count.
- `active_flag` is an `INT`.
- Bronze lineage is present.
- Any safe-cast failure is retained for later DQ; no row is deleted.


## 4. Build `silver_carriers_candidate`


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_carriers_candidate
USING DELTA
AS
SELECT
    UPPER(TRIM(carrier_code)) AS carrier_code,
    TRIM(carrier_name) AS carrier_name,
    TRY_CAST(active_flag AS INT) AS active_flag,

    -- Bronze lineage
    _source_row_number,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash,
    _rescued_payload,

    -- Silver Candidate metadata
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'skyops_silver_candidate_v1.0' AS _candidate_schema_version

FROM bronze_carriers;


In [ ]:
%sql
SELECT
    carrier_code,
    carrier_name,
    active_flag,
    _source_file_name,
    _ingestion_run_id,
    _bronze_record_hash
FROM silver_carriers_candidate
LIMIT 10;


In [ ]:
%sql
DESCRIBE silver_carriers_candidate;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM bronze_carriers) AS bronze_rows,
    (SELECT COUNT(*) FROM silver_carriers_candidate) AS candidate_rows,
    (SELECT COUNT(*) FROM silver_carriers_candidate)
      - (SELECT COUNT(*) FROM bronze_carriers) AS difference,
    CASE
      WHEN (SELECT COUNT(*) FROM bronze_carriers)
         = (SELECT COUNT(*) FROM silver_carriers_candidate)
      THEN 'PASS' ELSE 'CHECK'
    END AS status;


In [ ]:
%sql
SELECT COUNT(*) AS active_flag_parse_failures
FROM bronze_carriers
WHERE active_flag IS NOT NULL
  AND TRY_CAST(active_flag AS INT) IS NULL;


### Carriers checkpoint

The Candidate table must preserve every Bronze physical row and its lineage. No `_source_period` is referenced because the corrected Week-4 reference-source contract does not define it.


## 5. Build `silver_routes_candidate`


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_routes_candidate
USING DELTA
AS
SELECT
    UPPER(TRIM(route_id)) AS route_id,
    UPPER(TRIM(origin_airport_code)) AS origin_airport_code,
    UPPER(TRIM(destination_airport_code)) AS destination_airport_code,
    TRIM(route_label) AS route_label,
    TRY_CAST(distance_miles AS INT) AS distance_miles,
    TRIM(distance_band) AS distance_band,

    -- Bronze lineage
    _source_row_number,
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash,
    _rescued_payload,

    -- Silver Candidate metadata
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'skyops_silver_candidate_v1.0' AS _candidate_schema_version

FROM bronze_routes;


In [ ]:
%sql
SELECT
    route_id,
    origin_airport_code,
    destination_airport_code,
    route_label,
    distance_miles,
    distance_band,
    _source_file_name,
    _ingestion_run_id,
    _bronze_record_hash
FROM silver_routes_candidate
LIMIT 10;


In [ ]:
%sql
DESCRIBE silver_routes_candidate;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM bronze_routes) AS bronze_rows,
    (SELECT COUNT(*) FROM silver_routes_candidate) AS candidate_rows,
    (SELECT COUNT(*) FROM silver_routes_candidate)
      - (SELECT COUNT(*) FROM bronze_routes) AS difference,
    CASE
      WHEN (SELECT COUNT(*) FROM bronze_routes)
         = (SELECT COUNT(*) FROM silver_routes_candidate)
      THEN 'PASS' ELSE 'CHECK'
    END AS status;


In [ ]:
%sql
SELECT COUNT(*) AS distance_parse_failures
FROM bronze_routes
WHERE distance_miles IS NOT NULL
  AND TRY_CAST(distance_miles AS INT) IS NULL;


### Routes checkpoint

`distance_miles` is converted to `INT` because the supplied data dictionary defines route distance as an integer. This is a type conversion only; Week 6 owns the business-rule test for whether distance is positive and consistent with the route band.


## 6. Build `silver_flights_candidate`


### Flight identity rule

The source manifest identifies `source_record_key` as the stable physical-row key and states that the Silver layer creates `flight_id`. No separate composite formula is supplied in the project pack.

This corrected notebook therefore creates:

`flight_id = TRIM(source_record_key)`

while also retaining the original `source_record_key` for traceability. This avoids losing the physical source identifier.


In [ ]:
%sql
CREATE OR REPLACE TABLE silver_flights_candidate
USING DELTA
AS
SELECT
    -- Silver flight identity + source identity
    TRIM(source_record_key) AS flight_id,
    TRIM(source_record_key) AS source_record_key,
    TRY_CAST(flight_date AS DATE) AS flight_date,
    UPPER(TRIM(reporting_carrier)) AS reporting_carrier,
    TRY_CAST(flight_number AS INT) AS flight_number,
    UPPER(TRIM(tail_number)) AS tail_number,

    -- Airports
    UPPER(TRIM(origin_airport_code)) AS origin_airport_code,
    UPPER(TRIM(destination_airport_code)) AS destination_airport_code,

    -- Scheduled / actual times remain numeric HHMM values in Candidate
    TRY_CAST(scheduled_departure_hhmm AS INT) AS scheduled_departure_hhmm,
    TRY_CAST(actual_departure_hhmm AS INT) AS actual_departure_hhmm,

    -- Departure delay
    TRY_CAST(departure_delay_signed_minutes AS DOUBLE)
        AS departure_delay_signed_minutes,
    TRY_CAST(departure_delay_minutes AS DOUBLE)
        AS departure_delay_minutes,

    -- Arrival
    TRY_CAST(scheduled_arrival_hhmm AS INT) AS scheduled_arrival_hhmm,
    TRY_CAST(actual_arrival_hhmm AS INT) AS actual_arrival_hhmm,

    TRY_CAST(arrival_delay_signed_minutes AS DOUBLE)
        AS arrival_delay_signed_minutes,
    TRY_CAST(arrival_delay_minutes AS DOUBLE)
        AS arrival_delay_minutes,

    -- Status
    TRY_CAST(cancelled_flag AS INT) AS cancelled_flag,
    NULLIF(TRIM(cancellation_code), '') AS cancellation_code,
    TRY_CAST(diverted_flag AS INT) AS diverted_flag,

    -- Duration fields
    TRY_CAST(scheduled_elapsed_minutes AS DOUBLE)
        AS scheduled_elapsed_minutes,
    TRY_CAST(actual_elapsed_minutes AS DOUBLE)
        AS actual_elapsed_minutes,
    TRY_CAST(air_time_minutes AS DOUBLE) AS air_time_minutes,
    TRY_CAST(taxi_out_minutes AS DOUBLE) AS taxi_out_minutes,
    TRY_CAST(taxi_in_minutes AS DOUBLE) AS taxi_in_minutes,

    -- Distance
    TRY_CAST(distance_miles AS DOUBLE) AS distance_miles,

    -- Delay causes
    TRY_CAST(carrier_delay_minutes AS DOUBLE)
        AS carrier_delay_minutes,
    TRY_CAST(weather_delay_minutes AS DOUBLE)
        AS weather_delay_minutes,
    TRY_CAST(nas_delay_minutes AS DOUBLE)
        AS nas_delay_minutes,
    TRY_CAST(security_delay_minutes AS DOUBLE)
        AS security_delay_minutes,
    TRY_CAST(late_aircraft_delay_minutes AS DOUBLE)
        AS late_aircraft_delay_minutes,

    -- Source provenance supplied by flights.csv
    TRIM(source_period) AS source_period,
    TRIM(source_original_filename) AS source_original_filename,
    TRY_CAST(source_row_number AS INT) AS source_row_number,

    -- Bronze lineage
    _source_file_name,
    _source_file_path,
    _ingested_at,
    _ingestion_run_id,
    _schema_version AS _bronze_schema_version,
    _record_hash AS _bronze_record_hash,
    _rescued_payload,

    -- Silver Candidate metadata
    CURRENT_TIMESTAMP() AS _candidate_created_at,
    'skyops_silver_candidate_v1.0' AS _candidate_schema_version

FROM bronze_flights;


In [ ]:
%sql
SELECT
    flight_id,
    source_record_key,
    flight_date,
    reporting_carrier,
    flight_number,
    origin_airport_code,
    destination_airport_code,
    scheduled_departure_hhmm,
    departure_delay_minutes,
    cancelled_flag,
    diverted_flag,
    _source_file_name,
    _ingestion_run_id,
    _bronze_record_hash
FROM silver_flights_candidate
LIMIT 10;


In [ ]:
%sql
DESCRIBE silver_flights_candidate;


In [ ]:
%sql
SELECT
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT flight_id) AS distinct_flight_ids,
    SUM(CASE WHEN flight_id IS NULL THEN 1 ELSE 0 END) AS null_flight_ids
FROM silver_flights_candidate;


### Flight identity checkpoint

The `flight_id` uniqueness test is an **observation for Week 6**, not a quarantine step in Week 5. The supplied DQ rule DQ-FLT-001 owns the business decision about uniqueness/conflicting duplicates.


## 7. Check safe-cast failures without deleting rows


In [ ]:
%sql
SELECT
    SUM(CASE
          WHEN flight_date IS NOT NULL
           AND TRY_CAST(flight_date AS DATE) IS NULL
          THEN 1 ELSE 0
        END) AS flight_date_parse_failures,

    SUM(CASE
          WHEN flight_number IS NOT NULL
           AND TRY_CAST(flight_number AS INT) IS NULL
          THEN 1 ELSE 0
        END) AS flight_number_parse_failures,

    SUM(CASE
          WHEN scheduled_departure_hhmm IS NOT NULL
           AND TRY_CAST(scheduled_departure_hhmm AS INT) IS NULL
          THEN 1 ELSE 0
        END) AS scheduled_departure_parse_failures,

    SUM(CASE
          WHEN departure_delay_minutes IS NOT NULL
           AND TRY_CAST(departure_delay_minutes AS DOUBLE) IS NULL
          THEN 1 ELSE 0
        END) AS departure_delay_parse_failures,

    SUM(CASE
          WHEN distance_miles IS NOT NULL
           AND TRY_CAST(distance_miles AS DOUBLE) IS NULL
          THEN 1 ELSE 0
        END) AS distance_parse_failures
FROM bronze_flights;


These are actual parse-failure counts from the Bronze data. A non-zero result is retained as evidence for later DQ. Week 5 does not repair or remove the affected record.


## 8. Validate row counts for all four Candidate tables


In [ ]:
%sql
WITH counts AS (
    SELECT
        'airports' AS entity,
        (SELECT COUNT(*) FROM bronze_airports) AS bronze_rows,
        (SELECT COUNT(*) FROM silver_airports_candidate) AS candidate_rows
    UNION ALL
    SELECT
        'carriers',
        (SELECT COUNT(*) FROM bronze_carriers),
        (SELECT COUNT(*) FROM silver_carriers_candidate)
    UNION ALL
    SELECT
        'routes',
        (SELECT COUNT(*) FROM bronze_routes),
        (SELECT COUNT(*) FROM silver_routes_candidate)
    UNION ALL
    SELECT
        'flights',
        (SELECT COUNT(*) FROM bronze_flights),
        (SELECT COUNT(*) FROM silver_flights_candidate)
)
SELECT
    entity,
    bronze_rows,
    candidate_rows,
    candidate_rows - bronze_rows AS difference,
    CASE
      WHEN bronze_rows = candidate_rows THEN 'PASS'
      ELSE 'CHECK'
    END AS status
FROM counts
ORDER BY entity;


### Count checkpoint

Every entity should show `difference = 0` and `status = PASS`.

A mismatch means Week 5 accidentally changed the physical grain. Look for filters, `DISTINCT`, or grain-changing joins. This notebook intentionally uses no such operations.


## 9. Prove Bronze record-hash lineage


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM (
        SELECT _record_hash
        FROM bronze_airports
        EXCEPT ALL
        SELECT _bronze_record_hash
        FROM silver_airports_candidate
    )) AS bronze_rows_missing_in_candidate,
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash
        FROM silver_airports_candidate
        EXCEPT ALL
        SELECT _record_hash
        FROM bronze_airports
    )) AS unexpected_candidate_rows;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM (
        SELECT _record_hash
        FROM bronze_carriers
        EXCEPT ALL
        SELECT _bronze_record_hash
        FROM silver_carriers_candidate
    )) AS bronze_rows_missing_in_candidate,
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash
        FROM silver_carriers_candidate
        EXCEPT ALL
        SELECT _record_hash
        FROM bronze_carriers
    )) AS unexpected_candidate_rows;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM (
        SELECT _record_hash
        FROM bronze_routes
        EXCEPT ALL
        SELECT _bronze_record_hash
        FROM silver_routes_candidate
    )) AS bronze_rows_missing_in_candidate,
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash
        FROM silver_routes_candidate
        EXCEPT ALL
        SELECT _record_hash
        FROM bronze_routes
    )) AS unexpected_candidate_rows;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM (
        SELECT _record_hash
        FROM bronze_flights
        EXCEPT ALL
        SELECT _bronze_record_hash
        FROM silver_flights_candidate
    )) AS bronze_rows_missing_in_candidate,
    (SELECT COUNT(*) FROM (
        SELECT _bronze_record_hash
        FROM silver_flights_candidate
        EXCEPT ALL
        SELECT _record_hash
        FROM bronze_flights
    )) AS unexpected_candidate_rows;


### Lineage checkpoint

For each entity both values should be zero:

- `bronze_rows_missing_in_candidate = 0`
- `unexpected_candidate_rows = 0`

This proves the Candidate rows are traceable to the same physical Bronze records.


## 10. Check lineage completeness


In [ ]:
%sql
SELECT
  'airports' AS entity,
  SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_source_file,
  SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run_id,
  SUM(CASE WHEN _bronze_schema_version IS NULL THEN 1 ELSE 0 END) AS missing_schema_version,
  SUM(CASE WHEN _bronze_record_hash IS NULL THEN 1 ELSE 0 END) AS missing_record_hash
FROM silver_airports_candidate

UNION ALL

SELECT
  'carriers',
  SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_schema_version IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_record_hash IS NULL THEN 1 ELSE 0 END)
FROM silver_carriers_candidate

UNION ALL

SELECT
  'routes',
  SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_schema_version IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_record_hash IS NULL THEN 1 ELSE 0 END)
FROM silver_routes_candidate

UNION ALL

SELECT
  'flights',
  SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_schema_version IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN _bronze_record_hash IS NULL THEN 1 ELSE 0 END)
FROM silver_flights_candidate
ORDER BY entity;


### Lineage completeness checkpoint

Required lineage fields should have zero missing values for the controlled Week-4 Bronze output.

For flights, additionally verify that `source_record_key`, `source_original_filename` and `source_row_number` remain present because they are part of the source contract.


In [ ]:
%sql
SELECT
    SUM(CASE WHEN source_record_key IS NULL THEN 1 ELSE 0 END) AS missing_source_record_key,
    SUM(CASE WHEN source_original_filename IS NULL THEN 1 ELSE 0 END) AS missing_source_original_filename,
    SUM(CASE WHEN source_row_number IS NULL THEN 1 ELSE 0 END) AS missing_source_row_number
FROM silver_flights_candidate;


## 11. Confirm Candidate storage format


In [ ]:
%sql
SELECT
    table_name,
    data_source_format
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name IN (
      'silver_airports_candidate',
      'silver_carriers_candidate',
      'silver_routes_candidate',
      'silver_flights_candidate'
  )
ORDER BY table_name;


Expected result: all four Candidate tables are persistent Delta tables.

If `system.information_schema.tables` is unavailable in your Databricks edition, run `DESCRIBE DETAIL <table_name>` for each Candidate table.


## 12. Inspect the final Candidate schemas


In [ ]:
%sql
DESCRIBE silver_airports_candidate;


In [ ]:
%sql
DESCRIBE silver_carriers_candidate;


In [ ]:
%sql
DESCRIBE silver_routes_candidate;


In [ ]:
%sql
DESCRIBE silver_flights_candidate;


### Schema checkpoint

Confirm that:

- identifiers/codes are strings;
- documented numeric fields have the intended numeric types;
- `flight_date` is `DATE`;
- `flight_id` and `source_record_key` are both present for flights;
- Bronze lineage fields are present;
- Candidate metadata fields are present.

Do not add unsupported calculated fields merely to make the schema look more complex.


## 13. Controlled repeat-run test


The four Candidate writes use `CREATE OR REPLACE TABLE`, matching the controlled snapshot approach in the supplied Week-5 learning material.

To prove rerun behavior:

1. record the current Candidate counts;
2. rerun only the four `CREATE OR REPLACE TABLE` cells;
3. run the reconciliation cell again;
4. confirm the counts are unchanged.

A successful SQL command alone is not proof of rerun safety.


In [ ]:
%sql
SELECT
    'airports' AS entity, COUNT(*) AS candidate_rows
FROM silver_airports_candidate
UNION ALL
SELECT 'carriers', COUNT(*) FROM silver_carriers_candidate
UNION ALL
SELECT 'routes', COUNT(*) FROM silver_routes_candidate
UNION ALL
SELECT 'flights', COUNT(*) FROM silver_flights_candidate
ORDER BY entity;


### Repeat-run evidence

After rerunning the four write cells, execute the count reconciliation in Section 8 again. The result should remain `PASS` with zero differences.


## 14. Inspect Delta history


In [ ]:
%sql
DESCRIBE HISTORY silver_airports_candidate
LIMIT 5;


In [ ]:
%sql
DESCRIBE HISTORY silver_carriers_candidate
LIMIT 5;


In [ ]:
%sql
DESCRIBE HISTORY silver_routes_candidate
LIMIT 5;


In [ ]:
%sql
DESCRIBE HISTORY silver_flights_candidate
LIMIT 5;


Delta history should show the controlled table writes. Use the history as evidence that the Candidate tables are persistent Delta assets and that a rerun creates a new table version rather than silently appending duplicate physical rows.


## 15. What Week 5 intentionally does NOT do


The following are deliberately excluded from this notebook:

- no DQ quarantine;
- no Trusted Silver;
- no deletion of failed rows;
- no duplicate resolution;
- no route/carrier/airport relationship enforcement;
- no HHMM business-rule validation;
- no delay consistency enforcement;
- no cancellation/diversion enforcement;
- no delay-cause reconciliation;
- no route-distance business-rule enforcement;
- no Gold KPI aggregation;
- no Power BI work;
- no streaming/NDJSON processing.

Those rules belong to later stages in the supplied project plan, especially Week 6 DQ and Week 10 streaming.


## 16. Project-to-material mapping


| Week-5 learning-material pattern | SkyOps implementation |
|---|---|
| Bronze → Silver Candidate | four SkyOps Bronze tables → four Candidate tables |
| Standardise identifiers | `UPPER(TRIM(...))` on approved codes |
| Safe typing | `TRY_CAST` on documented target fields |
| Calculated field | only `flight_id` is created because the project identifies a Silver business key but gives no additional KPI formula |
| Preserve physical rows | no filters, `DISTINCT` or grain-changing joins |
| Preserve lineage | source file/path, run ID, schema version, Bronze hash and rescued context |
| Validate | count reconciliation + record-hash comparison + schema/metadata checks |
| Rerun | `CREATE OR REPLACE TABLE` + stable count proof |


## 17. Final Week-5 checklist


- [ ] Correct catalog/schema confirmed.
- [ ] All four Week-4 Bronze tables exist.
- [ ] Bronze tables were not modified.
- [ ] Four Silver Candidate Delta tables were created.
- [ ] Only project-supported standardisation and type conversion were used.
- [ ] `flight_id` and original `source_record_key` are both retained.
- [ ] `flight_date` is safely converted to `DATE`.
- [ ] Numeric and flag fields use `TRY_CAST`.
- [ ] All Bronze lineage fields are preserved.
- [ ] Flight source provenance fields are preserved.
- [ ] Bronze count equals Candidate count for every entity.
- [ ] Bronze record hashes match Candidate lineage hashes.
- [ ] Safe-cast failures remain visible.
- [ ] No rows were quarantined or deleted.
- [ ] Controlled rerun keeps counts stable.
- [ ] Delta history was inspected.
- [ ] Actual Databricks results, not invented values, are used as evidence.


## Week-5 exit point

The pipeline now has:

```text
Approved source files
        ↓
Bronze Delta
        ↓
Silver Candidate Delta
        ↓
Week-6 Data Quality
        ↓
Trusted Silver / Quarantine
        ↓
Gold
```

**Stop here for Week 5.**

Week 6 should apply the documented DQ rules and decide which Candidate records become Trusted Silver and which require quarantine.


## AI Transparency Note

AI was used to help compare the supplied Week-5 learning material, the SkyOps project data dictionary/source manifest, the Week-4 Bronze contract, and the submitted Week-5 notebook.

The notebook was deliberately constrained to transformations supported by those materials. In particular, no unsupported KPI formulas, business-rule corrections, quarantine logic, or reference-enrichment rules were invented.

Before submission, the student team must execute the notebook in its own Databricks workspace and verify every schema, count, lineage result and Delta-history result.
